In [ ]:
# Imports

import torch
import torch.nn as nn
import torch.optim as optim
from tqdm.autonotebook import tqdm
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import matplotlib.pyplot as plt
import numpy as np

<ipython-input-3-99b39718beb4>:6: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [ ]:
# for temporary use, while we finish data processing
!pip install datasets
from datasets import load_dataset

ds = load_dataset("stochastic/random_streetview_images_pano_v0.0.2")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 17.8 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2024.12.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which 

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/6.85k [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/974 [00:00<?, ?B/s]

(…)-00000-of-00006-9089804f14aaccce.parquet:   0%|          | 0.00/472M [00:00<?, ?B/s]

...

(…)-00002-of-00006-f8ce946916e330c8.parquet:   0%|          | 0.00/469M [00:00<?, ?B/s]

(…)-00003-of-00006-f8e97309feb01a88.parquet:   0%|          | 0.00/466M [00:00<?, ?B/s]

(…)-00004-of-00006-450fd3815b925113.parquet:   0%|          | 0.00/467M [00:00<?, ?B/s]

(…)-00005-of-00006-a0dd1c677754786b.parquet:   0%|          | 0.00/465M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/11054 [00:00<?, ? examples/s]

In [ ]:
# convert the lat and long to float
ds = ds.map(lambda x: {'latitude': float(x['latitude']), 'longitude': float(x['longitude'])})

Map:   0%|          | 0/11054 [00:00<?, ? examples/s]

In [ ]:
class HaversineLoss(torch.nn.Module):
    def __init__(self):
        super(HaversineLoss, self).__init__()
        # Earth's radius in km (use 3956 for miles)
        self.R = torch.tensor(6371.0, requires_grad=False)

    def forward(self, preds, targets):
        # Ensure R is on the same device as inputs
        self.R = self.R.to(preds.device)

        # Convert degrees to radians
        preds_rad = torch.deg2rad(preds)
        targets_rad = torch.deg2rad(targets)

        # Split into latitude and longitude
        pred_lat, pred_lon = preds_rad[:, 0], preds_rad[:, 1]
        target_lat, target_lon = targets_rad[:, 0], targets_rad[:, 1]

        # Calculate differences
        dlat = target_lat - pred_lat
        dlon = target_lon - pred_lon

        # Haversine formula
        a = (torch.sin(dlat / 2) ** 2 +
             torch.cos(pred_lat) * torch.cos(target_lat) * torch.sin(dlon / 2) ** 2)
        c = 2 * torch.asin(torch.sqrt(a))

        # Calculate distance
        distances = c * self.R

        # Return mean distance as loss
        return torch.mean(distances)


## ViT MODEL

In [ ]:
!pip install transformers

In [ ]:
!pip install --upgrade torch torchvision


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 108.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 83.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 92.6 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitli

In [ ]:
from transformers import ViTModel, ViTFeatureExtractor

class ViTRegressor(nn.Module):
    def __init__(self, model_name="google/vit-base-patch16-224-in21k"):
        super(ViTRegressor, self).__init__()
        # Load pre-trained ViT model
        self.vit = ViTModel.from_pretrained(model_name)
        # Add a dropout layer for regularization
        self.dropout = nn.Dropout(0.1)
        # Regression head: outputs 2 values (latitude and longitude)
        self.regressor = nn.Linear(self.vit.config.hidden_size, 2)

    def forward(self, pixel_values):
        outputs = self.vit(pixel_values=pixel_values)
        # Use the [CLS] token representation (pooler output)
        pooled_output = outputs.pooler_output
        pooled_output = self.dropout(pooled_output)
        regression_output = self.regressor(pooled_output)
        return regression_output

# Initialize model and feature extractor
model = ViTRegressor()
feature_extractor = ViTFeatureExtractor.from_pretrained("google/vit-base-patch16-224-in21k", size=384)

NameError: name 'nn' is not defined

In [ ]:
##DONT NEED THIS - ALL DONE

# Function to split each image into 3
def split_image_into_three_tensor(image_tensor: torch.Tensor, target_size=(384, 384)):
    C, H, W = image_tensor.shape
    third_width = W // 3

    # Split into 3 along width (dim=2)
    splits = [
        image_tensor[:, :, i*third_width : (i+1)*third_width]  # Shape [C, H, W/3]
        for i in range(3)
    ]

    # Resize each split to target_size
    resized_splits = [
        torch.nn.functional.interpolate(split.unsqueeze(0), size=target_size, mode='bilinear')[0]
        for split in splits
    ]

    return resized_splits

# Apply to all images
new_dataset = []

# IF WANTING TO TEST ON FIRST FEW IMAGES: ADD [:500]
sample = ds['train']

for i in range(len(sample['image'])):
    # Extract the image tensor, latitude, and longitude
    image_tensor = sample['image'][i]  # Shape: [C, H, W]
    latitude = sample['latitude'][i]
    longitude = sample['longitude'][i]

    # Split the image into three parts and resize them
    split_images = split_image_into_three_tensor(image_tensor, (384, 384))

    # Create new data points for each split
    for split_image in split_images:
        new_data_point = {
            'image': split_image,  # Resized split image tensor
            'latitude': latitude,  # Same latitude as the original
            'longitude': longitude  # Same longitude as the original
        }
        new_dataset.append(new_data_point)

# Create a new dataset from the processed splits
from datasets import Dataset
new_dataset = Dataset.from_dict({
    'image': [data['image'] for data in new_dataset],
    'latitude': [data['latitude'] for data in new_dataset],
    'longitude': [data['longitude'] for data in new_dataset]
})

# Apply the transformation to the new dataset
def transform(example):
    inputs = feature_extractor(example['image'], return_tensors="pt")
    example["pixel_values"] = inputs["pixel_values"][0]
    return example

new_dataset = new_dataset.map(transform)


Map:   0%|          | 0/11054 [00:00<?, ? examples/s]

In [ ]:
#Convert pixels into tensors

# Collate function for batching
def collate_fn(batch):
    pixel_values = []
    for item in batch:
        pixel_tensor = torch.as_tensor(item["pixel_values"])
        pixel_values.append(pixel_tensor)

    pixel_values = torch.stack(pixel_values)
    targets = torch.tensor([[item["latitude"], item["longitude"]] for item in batch], dtype=torch.float32)
    return {"pixel_values": pixel_values, "targets": targets}

# Create a DataLoader for the new dataset
train_loader = DataLoader(new_dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)


In [ ]:
# Initialize the model, loss function, optimizer
model = ViTRegressor()
criterion = HaversineLoss()
optimizer = optim.Adam(model.parameters(), lr=2e-5)

# Move the model to the appropriate device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Training loop
num_epochs = 5  # Adjust the number of epochs as needed
for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    for batch in train_loader:
        pixel_values = batch["pixel_values"].to(device)
        targets = batch["targets"].to(device)

        optimizer.zero_grad()
        outputs = model(pixel_values)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}")


Epoch [1/5], Loss: 5945.4948


KeyboardInterrupt: 

In [ ]:
# Imports
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import numpy as np
from tqdm.autonotebook import tqdm
from torch.utils.data import DataLoader
from datasets import load_from_disk
print("All imports successful.")

# Create Haversine Loss Function
class HaversineLoss(torch.nn.Module):
    def __init__(self):
        super(HaversineLoss, self).__init__()
        # Earth's radius in km (use 3956 for miles)
        self.R = torch.tensor(6371.0, requires_grad=False)

    def forward(self, preds, targets):
        # Ensure R is on the same device as inputs
        self.R = self.R.to(preds.device)

        # Convert degrees to radians
        preds_rad = torch.deg2rad(preds)
        targets_rad = torch.deg2rad(targets)

        # Split into latitude and longitude
        pred_lat, pred_lon = preds_rad[:, 0], preds_rad[:, 1]
        target_lat, target_lon = targets_rad[:, 0], targets_rad[:, 1]

        # Calculate differences
        dlat = target_lat - pred_lat
        dlon = target_lon - pred_lon

        # Haversine formula
        a = (torch.sin(dlat/2)**2 +
             torch.cos(pred_lat) * torch.cos(target_lat) * torch.sin(dlon/2)**2)
        c = 2 * torch.asin(torch.sqrt(a))

        # Calculate distance
        distances = c * self.R

        # Return mean distance as loss
        return torch.mean(distances)

print("Haversine Loss Function created successfully.")

# Load the dataset
imgs = load_from_disk('processed_streetview_three_eight_four_tensor')
print("Dataset loaded successfully.")
print("Dataset length:", len(imgs))
print("Dataset keys:", imgs[0].keys())
print("Img dtype:", type(imgs[0]['image']))
#print(imgs[0]['image'][0][0])

# Create model specification
# Use mobilenet v3 as the model, without pretrained weights
model = torch.hub.load('pytorch/vision', 'mobilenet_v3_small', weights = None)
# replace the head to output 2 numbers for regression
model.classifier[3] = nn.Linear(in_features=1024, out_features=2, bias=True)
print("Model created successfully.")

# Test with one image
#example_image = imgs[0]['image']
#output = model(example_image.float().unsqueeze(0))
#print("Test Output Latitude and Longitude:", output)

# Move to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print('Model moved to device.')

# Define the loss function and optimizer
optimizer = optim.AdamW(model.parameters(), lr=0.01)
criterion = HaversineLoss()

# Create DataLoader
dataloader = DataLoader(imgs, batch_size=64, shuffle=True)
print('Dataloader created.')

# Training loop
num_epochs = 20
for epoch in range(num_epochs):
    epoch_loss = 0.0
    for batch in tqdm(dataloader, desc=f"Epoch {epoch + 1}/{num_epochs}"):
        optimizer.zero_grad()
        images = torch.from_numpy(np.asarray(batch['image']))
        images = torch.permute(images, (3, 0, 1, 2)).float()
        images = images.to(device)
        targets = torch.stack((torch.tensor(batch['latitude']), torch.tensor(batch['longitude'])), dim=1)
        targets = targets.to(device)

        # Forward pass
        outputs = model(images)

        # Calculate the loss
        loss = criterion(outputs, targets)

        # Backpropagation and optimization
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss/len(dataloader)}")
    # save the model every epoch
    torch.save(model.state_dict(), f'cnn_scratch_epoch_{epoch+1}.pth')
    print(f"Model saved for epoch {epoch+1}")


print('Training Complete.')

# save the model
torch.save(model.state_dict(), 'cnn_scratch_trained.pth')
print("Model saved successfully.")